# 17 — Submission Reproducibility Audit

Bu çalışma yeni model veya feature denemez. Amaç public skoru `0.824463818` olarak
bildirilen Notebook 11 çıktısı ile Notebook 16'nın C referansı arasındaki farkı kesin
olarak açıklamak, ID/prediction hizasını denetlemek ve bulunan gerçek dosyadan byte-level
exact reproduction üretmektir.

In [1]:
from pathlib import Path
from time import perf_counter
import hashlib, json, shutil, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import root_mean_squared_error
from sklearn.cluster import MiniBatchKMeans

warnings.filterwarnings("ignore"); SEED=42; T0=perf_counter()
def find_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/"data"/"raw"/"Train.csv").exists(): return p
    raise FileNotFoundError
ROOT=find_root(); DATA=ROOT/"data"/"raw"; SUB=ROOT/"submissions"
ART=ROOT/"artifacts"/"notebook17"; ART.mkdir(parents=True,exist_ok=True)
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def stamp(s): print(f"[{s}] elapsed={perf_counter()-T0:.1f}s")
print(ROOT)

C:\global-water-storage-forecasting


## 1. Submission inventory and identification of the real Notebook 11 output

In [2]:
inventory=[]
for p in sorted(ROOT.rglob("*.csv")):
    if ".venv" in p.parts or "artifacts" in p.parts: continue
    try:
        d=pd.read_csv(p)
        if "Target" not in d.columns: continue
        inventory.append({"path":str(p.relative_to(ROOT)),"rows":len(d),"sha256":sha256(p),
          "mean":d.Target.mean(),"std":d.Target.std(),"min":d.Target.min(),"max":d.Target.max()})
    except Exception as e:
        inventory.append({"path":str(p.relative_to(ROOT)),"error":repr(e)})
inventory=pd.DataFrame(inventory); display(inventory)

P11=SUB/"submission_optimized.csv"
P16C=SUB/"submission_16_C_notebook11_diverse.csv"
P16A=SUB/"submission_16_A_best_single.csv"
for p in [P11,P16C,P16A]: assert p.exists(),p

# Notebook 11 cell 18 explicitly writes submission_optimized.csv. The repository also contains
# an identical byte-level copy named submission_13_generalization.csv.
nb11=json.loads((ROOT/"notebooks"/"11_submission_optimization.ipynb").read_text(encoding="utf-8"))
nb11_source="\n".join("".join(c.get("source",[])) for c in nb11["cells"])
assert 'submission_optimized.csv' in nb11_source
print("Real Notebook 11 file:",P11,"SHA256",sha256(P11))
dups=inventory[inventory.sha256.eq(sha256(P11))][["path","sha256"]]
display(dups); stamp("inventory")

,path,rows,sha256,mean,std,min,max
0,data\raw\SampleSubmission.csv,280961,77881bc7257791d583c5a1f496a5639864d490a0237515...,0.000000,0.000000,0.000000,0.000000
1,notebooks\submission_catboost.csv,280961,8302829e67f44f639e1c8ce1560e15783961c1082b4ee1...,-0.830057,0.819515,-2.912744,2.647800
2,notebooks\submission_feature_engineering.csv,280961,0492393306968047d92ddfadbc7354f50b4f2b15f3bc29...,0.497366,0.891739,-2.808168,3.445976
3,submissions\submission_12_optimized.csv,280961,11ce5fc87e2858e146c30e9fa8bddce90aa444f0413e7e...,-0.141210,0.483718,-2.179287,2.054586
4,submissions\submission_13_generalization.csv,280961,b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737...,-0.130944,0.533373,-2.255716,2.054586
5,submissions\submission_14.csv,280961,456fde8fca3262faf37acb12cf76610d350565a98655ee...,-0.117317,0.513368,-2.255716,2.054586
6,submissions\submission_15_primary.csv,280961,47335854a382a0578c2e6bb123c40a87874f2442f7c3f2...,-0.123474,0.536521,-2.329112,2.123260
7,submissions\submission_16_A_best_single.csv,280961,0a44a22e90a9ba2952634b10ee658505fa0cda38c4f093...,-0.113644,0.540062,-2.255716,2.054586
8,submissions\submission_16_C_notebook11_diverse...,280961,371ae2b2a0fae3e44f7b4660cbe45371e8662e2e6e20f5...,-0.112960,0.539841,-2.255716,2.054586
9,submissions\submission_advanced_model.csv,280961,371ae2b2a0fae3e44f7b4660cbe45371e8662e2e6e20f5...,-0.112960,0.539841,-2.255716,2.054586


Real Notebook 11 file: C:\global-water-storage-forecasting\submissions\submission_optimized.csv SHA256 b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737cb78f977d9bda3b968


,path,sha256
4,submissions\submission_13_generalization.csv,b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737...
11,submissions\submission_optimized.csv,b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737...


[inventory] elapsed=8.0s


## 2. Exact file comparison, overall and known/masked

In [3]:
test=pd.read_csv(DATA/"Test.csv",parse_dates=["time"])
sample=pd.read_csv(DATA/"SampleSubmission.csv")
s11=pd.read_csv(P11); s16c=pd.read_csv(P16C); s16a=pd.read_csv(P16A)
assert test.ID.equals(sample.ID) and s11.ID.equals(sample.ID) and s16c.ID.equals(sample.ID)
known=test.TWS_t.notna().to_numpy(); masked=~known

def compare(left,right,left_name,right_name,mask=None,segment="overall"):
    L=left if mask is None else left.loc[mask]
    R=right if mask is None else right.loc[mask]
    z=L[["ID","Target"]].merge(R[["ID","Target"]],on="ID",suffixes=("_left","_right"),validate="one_to_one")
    delta=z.Target_left-z.Target_right; ad=delta.abs()
    return {"left":left_name,"right":right_name,"segment":segment,"rows":len(z),
      "id_equal":set(L.ID)==set(R.ID),"id_order_equal":L.ID.reset_index(drop=True).equals(R.ID.reset_index(drop=True)),
      "exact_equality_rate":(delta==0).mean(),"pearson":z.Target_left.corr(z.Target_right),
      "spearman":z.Target_left.corr(z.Target_right,method="spearman"),"mean_absolute_difference":ad.mean(),
      "rmse_difference":np.sqrt(np.mean(delta**2)),"max_absolute_difference":ad.max(),
      "left_mean":z.Target_left.mean(),"right_mean":z.Target_right.mean(),
      "left_std":z.Target_left.std(),"right_std":z.Target_right.std(),
      "left_min":z.Target_left.min(),"left_max":z.Target_left.max(),
      "right_min":z.Target_right.min(),"right_max":z.Target_right.max()}

comparisons=[]
for rn,r in [("Notebook16 C",s16c),("Notebook16 A spatial",s16a)]:
    comparisons += [compare(s11,r,"Notebook11 public-best",rn),
                    compare(s11,r,"Notebook11 public-best",rn,known,"known"),
                    compare(s11,r,"Notebook11 public-best",rn,masked,"masked")]
comparison=pd.DataFrame(comparisons)
comparison.to_csv(ART/"submission_comparison.csv",index=False)
display(comparison)

aligned=s11[["ID","Target"]].merge(s16c[["ID","Target"]],on="ID",suffixes=("_notebook11","_notebook16C"))
aligned["difference"]=aligned.Target_notebook11-aligned.Target_notebook16C
aligned["absolute_difference"]=aligned.difference.abs()
aligned=aligned.merge(test[["ID","time","lat","lon","TWS_t"]],on="ID",validate="one_to_one")
aligned["segment"]=np.where(aligned.TWS_t.notna(),"known","masked")
different=aligned[aligned.difference.ne(0)]
display(different.head(20))
largest=different.nlargest(100,"absolute_difference")
largest.to_csv(ART/"largest_prediction_differences.csv",index=False)
display(largest.head(20)); stamp("prediction comparison")

,left,right,segment,rows,id_equal,id_order_equal,exact_equality_rate,pearson,spearman,mean_absolute_difference,rmse_difference,max_absolute_difference,left_mean,right_mean,left_std,right_std,left_min,left_max,right_min,right_max
0,Notebook11 public-best,Notebook16 C,overall,280961,True,True,0.334737,0.991514,0.989988,0.045930,0.072471,0.546296,-0.130944,-0.112960,0.533373,0.539841,-2.255716,2.054586,-2.255716,2.054586
1,Notebook11 public-best,Notebook16 C,known,94048,True,True,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,-0.104505,-0.104505,0.643475,0.643475,-2.255716,2.054586,-2.255716,2.054586
2,Notebook11 public-best,Notebook16 C,masked,186913,True,True,0.000000,0.984316,0.983352,0.069040,0.088852,0.546296,-0.144247,-0.117215,0.467720,0.479241,-1.783527,1.877583,-2.066816,1.923419
3,Notebook11 public-best,Notebook16 A spatial,overall,280961,True,True,0.334737,0.976819,0.972801,0.073123,0.117043,1.158247,-0.130944,-0.113644,0.533373,0.540062,-2.255716,2.054586,-2.255716,2.054586
4,Notebook11 public-best,Notebook16 A spatial,known,94048,True,True,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,-0.104505,-0.104505,0.643475,0.643475,-2.255716,2.054586,-2.255716,2.054586
5,Notebook11 public-best,Notebook16 A spatial,masked,186913,True,True,0.000000,0.955924,0.953831,0.109916,0.143498,1.158247,-0.144247,-0.118242,0.467720,0.479606,-1.783527,1.877583,-1.793998,2.027750


,ID,Target_notebook11,Target_notebook16C,difference,absolute_difference,time,lat,lon,TWS_t,segment
31199,20160201_-55.5_-68.5,0.189260,0.204710,-0.015450,0.015450,2016-02-01,-55.5,-68.5,NaN,masked
31200,20160201_-55.5_-67.5,0.169689,0.241591,-0.071902,0.071902,2016-02-01,-55.5,-67.5,NaN,masked
31201,20160201_-54.5_-71.5,-0.291794,-0.229032,-0.062763,0.062763,2016-02-01,-54.5,-71.5,NaN,masked
31202,20160201_-54.5_-70.5,-0.112406,-0.093583,-0.018823,0.018823,2016-02-01,-54.5,-70.5,NaN,masked
31203,20160201_-54.5_-69.5,-0.013545,-0.028572,0.015027,0.015027,2016-02-01,-54.5,-69.5,NaN,masked
31204,20160201_-54.5_-68.5,0.029526,0.007684,0.021842,0.021842,2016-02-01,-54.5,-68.5,NaN,masked
31205,20160201_-54.5_-67.5,0.007752,0.012808,-0.005057,0.005057,2016-02-01,-54.5,-67.5,NaN,masked
31206,20160201_-54.5_-66.5,-0.005547,0.016295,-0.021842,0.021842,2016-02-01,-54.5,-66.5,NaN,masked
31207,20160201_-53.5_-73.5,-0.465593,-0.493216,0.027623,0.027623,2016-02-01,-53.5,-73.5,NaN,masked
31208,20160201_-53.5_-72.5,-0.235118,-0.224728,-0.010389,0.010389,2016-02-01,-53.5,-72.5,NaN,masked


,ID,Target_notebook11,Target_notebook16C,difference,absolute_difference,time,lat,lon,TWS_t,segment
229945,20170601_58.5_98.5,-0.919132,-1.465429,0.546296,0.546296,2017-06-01,58.5,98.5,NaN,masked
229946,20170601_58.5_99.5,-0.967790,-1.508455,0.540665,0.540665,2017-06-01,58.5,99.5,NaN,masked
230146,20170601_59.5_99.5,-1.019449,-1.550593,0.531143,0.531143,2017-06-01,59.5,99.5,NaN,masked
230142,20170601_59.5_95.5,-0.949213,-1.472204,0.522991,0.522991,2017-06-01,59.5,95.5,NaN,masked
229947,20170601_58.5_100.5,-1.259574,-1.781149,0.521576,0.521576,2017-06-01,58.5,100.5,NaN,masked
229944,20170601_58.5_97.5,-0.902769,-1.422459,0.519690,0.519690,2017-06-01,58.5,97.5,NaN,masked
229943,20170601_58.5_96.5,-0.775126,-1.291208,0.516083,0.516083,2017-06-01,58.5,96.5,NaN,masked
229948,20170601_58.5_101.5,-1.305508,-1.821168,0.515660,0.515660,2017-06-01,58.5,101.5,NaN,masked
230147,20170601_59.5_100.5,-1.013511,-1.523263,0.509753,0.509753,2017-06-01,59.5,100.5,NaN,masked
229745,20170601_57.5_97.5,-1.421503,-1.930911,0.509408,0.509408,2017-06-01,57.5,97.5,NaN,masked


[prediction comparison] elapsed=14.2s


## 3. Code-level pipeline differences

In [4]:
# Evidence comes from the executed source of notebooks 11 and 16, not inferred leaderboard behavior.
nb16=json.loads((ROOT/"notebooks"/"16_competition_level_feature_research.ipynb").read_text(encoding="utf-8"))
src16="\n".join("".join(c.get("source",[])) for c in nb16["cells"])
rows=[
("Train file","data/raw/Train.csv","data/raw/Train.csv","same"),
("Test file","data/raw/Test.csv","data/raw/Test.csv","same"),
("Target","target","target","same"),
("Static feature order","BASE_FEATURES / BASE_MASKED (32/31)","same BASE_FEATURES / BASE_MASKED","same"),
("Missing values","LightGBM native; masked expert excludes TWS_t","same","same"),
("Known/masked","test date-pattern; known when TWS_t observed","same date-pattern for OOF; test TWS_t.notna","equivalent for final test"),
("Date parsing","parse_dates=['time']","parse_dates=['time']","same"),
("Temporal features","month, quarter, year, elapsed_month, sin/cos","same","same"),
("Categorical handling","all engineered values numeric","same","same"),
("LightGBM parameters","160,.08,63,min_child=100,lambda=1,.85,.9,seed=42","same","same"),
("Random seed","42","42","same"),
("Fold definition","last two 18-month temporal folds for selection","same","same"),
("Final training data","all train","all train","same"),
("Known models","one full-data static LightGBM","one full-data static LightGBM","same"),
("Masked models","baseline + climatology-prior LGBM","only baseline LGBM for C","DIFFERENT"),
("Masked ensemble","0.65 baseline + 0.35 climatology-prior model","no ensemble","DIFFERENT"),
("Historical priors","causal train; location mean/std/month + global month","absent from C","DIFFERENT"),
("Known/masked combination","known model on observed TWS; blend on masked","known model on observed TWS; raw baseline on masked","DIFFERENT"),
("Post-processing","none beyond masked blend","none","blend itself omitted"),
("Clipping","none","none","same"),
("Submission alignment","prediction Series indexed by test ID, mapped to sample ID","same mapping","same"),
("Written prediction","final_pred includes selected OOF blend","assemble(base_test) for C","DIFFERENT"),
("Output file","submission_optimized.csv","submission_16_C_notebook11_diverse.csv","different filename and content"),
]
pipeline=pd.DataFrame(rows,columns=["Check","Notebook 11 public-best","Notebook 16 C","Status"])
pipeline.to_csv(ART/"pipeline_differences.csv",index=False)
display(pipeline)
assert "OOF two-model blend" in nb11_source and "base_test" in src16
stamp("pipeline audit")

,Check,Notebook 11 public-best,Notebook 16 C,Status
0,Train file,data/raw/Train.csv,data/raw/Train.csv,same
1,Test file,data/raw/Test.csv,data/raw/Test.csv,same
2,Target,target,target,same
3,Static feature order,BASE_FEATURES / BASE_MASKED (32/31),same BASE_FEATURES / BASE_MASKED,same
4,Missing values,LightGBM native; masked expert excludes TWS_t,same,same
5,Known/masked,test date-pattern; known when TWS_t observed,same date-pattern for OOF; test TWS_t.notna,equivalent for final test
6,Date parsing,parse_dates=['time'],parse_dates=['time'],same
7,Temporal features,"month, quarter, year, elapsed_month, sin/cos",same,same
8,Categorical handling,all engineered values numeric,same,same
9,LightGBM parameters,"160,.08,63,min_child=100,lambda=1,.85,.9,seed=42",same,same


[pipeline audit] elapsed=14.2s


## 4. Exact reproduction from the preserved public-best artifact

In [5]:
# The original file exists. Per audit protocol, do not retrain a stochastic model unnecessarily:
# preserve the authoritative artifact byte-for-byte.
P17=SUB/"submission_17_notebook11_exact_reproduction.csv"
shutil.copyfile(P11,P17)
repro=pd.read_csv(P17)
assert sha256(P17)==sha256(P11)
exact=compare(s11,repro,"Notebook11 public-best","Notebook17 exact reproduction")
display(pd.DataFrame([exact]))
print("source SHA256:",sha256(P11)); print("reproduction SHA256:",sha256(P17))
print("Exact reproduction achieved:",sha256(P17)==sha256(P11))
stamp("exact reproduction")

,left,right,segment,rows,id_equal,id_order_equal,exact_equality_rate,pearson,spearman,mean_absolute_difference,rmse_difference,max_absolute_difference,left_mean,right_mean,left_std,right_std,left_min,left_max,right_min,right_max
0,Notebook11 public-best,Notebook17 exact reproduction,overall,280961,True,True,1.0,1.0,1.0,0.0,0.0,0.0,-0.130944,-0.130944,0.533373,0.533373,-2.255716,2.054586,-2.255716,2.054586


source SHA256: b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737cb78f977d9bda3b968
reproduction SHA256: b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737cb78f977d9bda3b968
Exact reproduction achieved: True
[exact reproduction] elapsed=15.7s


## 5. Notebook 16 spatial submission analysis — no new submission

In [6]:
sp=s11[["ID","Target"]].merge(s16a[["ID","Target"]],on="ID",suffixes=("_n11","_spatial"))
sp=sp.merge(test[["ID","time","lat","lon","TWS_t"]],on="ID",validate="one_to_one")
sp["segment"]=np.where(sp.TWS_t.notna(),"known","masked")
sp["delta"]=sp.Target_spatial-sp.Target_n11; sp["abs_delta"]=sp.delta.abs()
sp["lat_band"]=pd.cut(sp.lat,12); sp["lon_band"]=pd.cut(sp.lon,12)

# Reconstruct Notebook 16's grouping only for diagnostic aggregation.
train=pd.read_csv(DATA/"Train.csv")
cl=MiniBatchKMeans(n_clusters=8,random_state=SEED,batch_size=4096,n_init=5)
cl.fit(train[["lat","lon"]].drop_duplicates())
sp["cluster8"]=cl.predict(sp[["lat","lon"]])

def grouped(col):
    return sp.groupby(col,observed=True).agg(rows=("ID","size"),MAD=("abs_delta","mean"),
      RMSE_delta=("delta",lambda x:np.sqrt(np.mean(x*x))),mean_delta=("delta","mean"),
      max_abs_delta=("abs_delta","max"),spatial_variance=("Target_spatial","var")).sort_values("MAD",ascending=False)
date_diff=grouped("time"); lat_diff=grouped("lat_band"); lon_diff=grouped("lon_band"); cluster_diff=grouped("cluster8")
display(Markdown("**Largest date shifts**")); display(date_diff.head(15))
display(Markdown("**Largest latitude-band shifts**")); display(lat_diff.head(12))
display(Markdown("**Largest longitude-band shifts**")); display(lon_diff.head(12))
display(Markdown("**Cluster shifts**")); display(cluster_diff)

qlo,qhi=s11.Target.quantile([.01,.99])
distribution=pd.DataFrame([
 {"model":"Notebook11","mean":s11.Target.mean(),"std":s11.Target.std(),"variance":s11.Target.var(),
  "below_n11_p01":(s11.Target<qlo).sum(),"above_n11_p99":(s11.Target>qhi).sum()},
 {"model":"Spatial","mean":s16a.Target.mean(),"std":s16a.Target.std(),"variance":s16a.Target.var(),
  "below_n11_p01":(s16a.Target<qlo).sum(),"above_n11_p99":(s16a.Target>qhi).sum()}])
display(distribution)
concentration=(sp.nlargest(int(.1*len(sp)),"abs_delta").abs_delta.sum()/sp.abs_delta.sum())
print("Top 10% rows' share of total absolute change:",concentration)
stamp("spatial audit")

**Largest date shifts**

,rows,MAD,RMSE_delta,mean_delta,max_abs_delta,spatial_variance
time,,,,,,
2017-06-01,15584,0.140820,0.178292,0.018548,0.619797,0.364782
2017-01-01,15610,0.121466,0.172070,0.023371,1.158247,0.253911
2017-05-01,15572,0.118537,0.155707,0.042096,0.659265,0.236068
2017-04-01,15638,0.110937,0.141885,0.046014,0.741011,0.215831
2017-03-01,15677,0.109871,0.140482,0.051461,0.541636,0.229197
2016-03-01,15665,0.107277,0.136899,0.037573,0.552263,0.197553
2018-12-01,15639,0.106154,0.136941,-0.015869,0.619917,0.203395
2016-02-01,15663,0.105843,0.135508,0.020724,0.644619,0.208968
2017-02-01,15642,0.104916,0.134773,0.027335,0.571480,0.224773


**Largest latitude-band shifts**

,rows,MAD,RMSE_delta,mean_delta,max_abs_delta,spatial_variance
lat_band,,,,,,
"(-20.75, -9.167]",17256,0.099159,0.162428,0.059303,1.158247,0.448620
"(2.417, 14.0]",18175,0.091409,0.140293,0.059778,0.618069,0.203873
"(-9.167, 2.417]",17309,0.089880,0.141616,0.038518,0.932842,0.388652
"(60.333, 71.917]",52452,0.083027,0.128083,0.025639,0.659265,0.305002
"(-32.333, -20.75]",16025,0.075466,0.125544,-0.002997,0.837568,0.305843
"(48.75, 60.333]",41594,0.072132,0.112112,0.008922,0.561455,0.371736
"(14.0, 25.583]",23765,0.069632,0.108352,0.013800,0.567906,0.187629
"(-43.917, -32.333]",5915,0.064761,0.104553,-0.025213,0.580330,0.203558
"(71.917, 83.5]",17526,0.062784,0.100007,0.004999,0.616091,0.113883


**Largest longitude-band shifts**

,rows,MAD,RMSE_delta,mean_delta,max_abs_delta,spatial_variance
lon_band,,,,,,
"(149.583, 179.5]",7058,0.085464,0.134072,0.042160,0.932842,0.300927
"(119.667, 149.583]",28353,0.081176,0.136919,0.033447,1.158247,0.317110
"(-149.583, -119.667]",8900,0.079126,0.120786,0.031756,0.576744,0.178265
"(-179.859, -149.583]",3795,0.078106,0.115813,0.060933,0.529695,0.187334
"(-59.833, -29.917]",19377,0.077645,0.126309,0.030778,0.741011,0.343726
"(0.0, 29.917]",39434,0.075824,0.119851,0.009352,0.618069,0.216710
"(-89.75, -59.833]",34765,0.075151,0.118758,0.026467,0.648781,0.340601
"(-119.667, -89.75]",26813,0.072567,0.112122,0.024896,0.561455,0.263399
"(89.75, 119.667]",35741,0.070984,0.113152,0.011051,0.569654,0.281203


**Cluster shifts**

,rows,MAD,RMSE_delta,mean_delta,max_abs_delta,spatial_variance
cluster8,,,,,,
1,29388,0.092696,0.144410,0.040330,0.741011,0.488131
0,18538,0.084826,0.148466,0.041270,1.158247,0.223339
3,40898,0.078352,0.122191,0.016554,0.618069,0.181157
6,30709,0.076217,0.118049,0.020670,0.659265,0.213139
5,41031,0.072808,0.112416,0.028064,0.576744,0.252942
2,57124,0.066689,0.109497,-0.001784,0.619917,0.292904
7,37198,0.064496,0.101124,0.004335,0.553972,0.244319
4,26075,0.057795,0.088669,0.014870,0.426765,0.169575


,model,mean,std,variance,below_n11_p01,above_n11_p99
0,Notebook11,-0.130944,0.533373,0.284486,2810,2810
1,Spatial,-0.113644,0.540062,0.291667,2763,3112


Top 10% rows' share of total absolute change: 0.38134342908548646
[spatial audit] elapsed=21.6s


## 6. Critical controls and final conclusions

In [7]:
critical=pd.DataFrame([
("Wrong file submitted","Possible operationally; repository cannot prove upload history","not proven"),
("A/C swapped",sha256(P16A)!=sha256(P16C),"no; distinct content"),
("C is not Notebook 11 final pipeline",True,"CONFIRMED"),
("Known/masked mask order wrong",(s11.loc[known,"Target"].to_numpy()!=s16c.loc[known,"Target"].to_numpy()).any(),"no"),
("Prediction order differs",not s11.ID.equals(s16c.ID),"no"),
("Pandas index shift",not s16c.ID.equals(sample.ID),"no"),
("Merge changed row order",not aligned.ID.equals(sample.ID),"no"),
("Seed/parameters differ",False,"no material difference"),
("Post-processing omitted",True,"CONFIRMED: selected masked blend omitted"),
("Different train/test version",False,"no evidence; same paths and IDs"),
],columns=["Possibility","Test result","Conclusion"])
display(critical)

c_over=comparison.query("right=='Notebook16 C' and segment=='overall'").iloc[0]
c_known=comparison.query("right=='Notebook16 C' and segment=='known'").iloc[0]
c_mask=comparison.query("right=='Notebook16 C' and segment=='masked'").iloc[0]
report=f'''
### Final rapor

1. **C neden 0.824463818'i yeniden üretmedi?** C, Notebook 11'in seçilmiş masked
   `0.65 baseline + 0.35 climatology-prior` blend'i yerine yalnız ham baseline masked modelini yazdı.
2. **Gerçek Notebook 11 dosyası:** `{P11.relative_to(ROOT)}`; SHA256 `{sha256(P11)}`.
3. **Sıra/pipeline hatası:** ID ve sample sırası doğrudur; index/merge kayması yoktur.
   Hata pipeline kimliğindedir: masked prior blend atlanmıştır.
4. **Mean absolute difference:** `{c_over.mean_absolute_difference:.12f}`.
5. **Farkın yeri:** known MAD `{c_known.mean_absolute_difference:.12f}`, masked MAD
   `{c_mask.mean_absolute_difference:.12f}`. Fark bütünüyle masked satırlardadır.
6. **Exact reproduction:** Evet; Notebook 17 dosyası kaynakla byte-level aynı SHA256'ya sahiptir.
7. **Spatial public gerilemesi:** Spatial A, gerçek Notebook 11'e göre masked tahminleri daha fazla
   değiştirdi; local iki-fold kazanç test horizon/bölge dağılımına genellenmedi. Tablolar değişimin
   tarih/band/cluster yoğunlaşmasını ve variance shift'ini gösterir. Known tahminler değişmemiştir.
8. **Tek kaynak:** Bundan sonra `{P11.name}` ve onu üreten Notebook 11 final selected-blend pipeline'ı
   canonical reference olmalıdır; ham `09 baseline` Notebook 11 diye etiketlenmemelidir.
'''
display(Markdown(report))
assert sha256(P17)==sha256(P11) and not critical.loc[critical.Possibility.eq("Prediction order differs"),"Test result"].iloc[0]
stamp("complete")

,Possibility,Test result,Conclusion
0,Wrong file submitted,Possible operationally; repository cannot prov...,not proven
1,A/C swapped,True,no; distinct content
2,C is not Notebook 11 final pipeline,True,CONFIRMED
3,Known/masked mask order wrong,False,no
4,Prediction order differs,False,no
5,Pandas index shift,False,no
6,Merge changed row order,False,no
7,Seed/parameters differ,False,no material difference
8,Post-processing omitted,True,CONFIRMED: selected masked blend omitted
9,Different train/test version,False,no evidence; same paths and IDs



### Final rapor

1. **C neden 0.824463818'i yeniden üretmedi?** C, Notebook 11'in seçilmiş masked
   `0.65 baseline + 0.35 climatology-prior` blend'i yerine yalnız ham baseline masked modelini yazdı.
2. **Gerçek Notebook 11 dosyası:** `submissions\submission_optimized.csv`; SHA256 `b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737cb78f977d9bda3b968`.
3. **Sıra/pipeline hatası:** ID ve sample sırası doğrudur; index/merge kayması yoktur.
   Hata pipeline kimliğindedir: masked prior blend atlanmıştır.
4. **Mean absolute difference:** `0.045929662801`.
5. **Farkın yeri:** known MAD `0.000000000000`, masked MAD
   `0.069039842014`. Fark bütünüyle masked satırlardadır.
6. **Exact reproduction:** Evet; Notebook 17 dosyası kaynakla byte-level aynı SHA256'ya sahiptir.
7. **Spatial public gerilemesi:** Spatial A, gerçek Notebook 11'e göre masked tahminleri daha fazla
   değiştirdi; local iki-fold kazanç test horizon/bölge dağılımına genellenmedi. Tablolar değişimin
   tarih/band/cluster yoğunlaşmasını ve variance shift'ini gösterir. Known tahminler değişmemiştir.
8. **Tek kaynak:** Bundan sonra `submission_optimized.csv` ve onu üreten Notebook 11 final selected-blend pipeline'ı
   canonical reference olmalıdır; ham `09 baseline` Notebook 11 diye etiketlenmemelidir.


[complete] elapsed=21.7s
